# Deep Learning-Based Deepfake Detection Under Unseen Manipulation Techniques
### Transfer Learning with MobileNetV2, OpenCV, and TensorFlow/Keras

---
**Student Name:** [Your Name]  
**Roll Number / ID:** [Your Roll Number]  
**Course / Degree:** B.Tech / B.E. / M.S. in Computer Science & Engineering  
**Institution / University:** [Your University / College Name]  
**Supervisor / Guide:** [Prof. Guide Name]  
**Academic Year:** 2025 – 2026  

---
### Abstract
This notebook demonstrates the end-to-end training and evaluation of a convolutional neural network designed to detect facial manipulation and evaluate generalization against **unseen manipulation techniques**.

We implement a transfer learning architecture utilizing **MobileNetV2** pretrained on ImageNet. To prevent the prevalent flaw of data leakage, the data pipeline partitions samples strictly into disjoint subsets. The model undergoes two-phase training:
1. **Phase 1:** Feature extraction training on the custom classification head with the MobileNetV2 backbone frozen.
2. **Phase 2:** Fine-tuning of the upper 30 convolutional layers with a reduced learning rate ($10^{-5}$).

Finally, we assess empirical performance on both a **Standard Test Set** and an **Unseen Manipulation Set**, plotting confusion matrices, ROC curves, and providing an interactive image upload prediction cell for real-time demonstration.

## Step 1: Environment Setup & Hardware Acceleration Check
Google Colab offers free GPU acceleration (e.g. NVIDIA T4). Let's verify GPU availability and install required libraries.

In [ ]:
# Check allocated GPU
!nvidia-smi

# Install KaggleHub for programmatic dataset acquisition
!pip install -q kagglehub opencv-python-headless pillow matplotlib scikit-learn

In [ ]:
import os
import random
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn import metrics

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import MobileNetV2

# Reproducibility seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
print("GPU Devices Available:", tf.config.list_physical_devices('GPU'))

## Step 2: Dataset Acquisition via KaggleHub
We acquire the genuine human face dataset containing **5,890 real photographs** and **7,000 real deepfakes** programmatically via KaggleHub.

In [ ]:
import kagglehub

print("Downloading genuine human face deepfake dataset via KaggleHub...")
dataset_root = kagglehub.dataset_download("kshitizbhargava/deepfake-face-images")
dataset_path = Path(dataset_root) / "Final Dataset"

real_source_dir = dataset_path / "Real"
fake_source_dir = dataset_path / "Fake"

all_real_files = sorted(list(real_source_dir.glob("*.jpg")) + list(real_source_dir.glob("*.png")))
all_fake_files = sorted(list(fake_source_dir.glob("*.jpg")) + list(fake_source_dir.glob("*.png")))

print(f"\nDataset Successfully Retrieved:")
print(f"  - Total Authentic Real Faces : {len(all_real_files):,}")
print(f"  - Total Manipulated Deepfakes : {len(all_fake_files):,}")

## Step 3: Leakage-Free Dataset Splitting
A major research contribution of our protocol is **preventing data leakage**.
We sample balanced subsets into:
- **Training Set:** 800 images (400 REAL, 400 FAKE)
- **Validation Set:** 200 images (100 REAL, 100 FAKE)
- **Standard Test Set:** 200 images (100 REAL, 100 FAKE)
- **Unseen Manipulation Set:** 200 images (100 REAL, 100 FAKE)

All sets are strictly disjoint: $\text{Train} \cap \text{Val} = \emptyset$, $\text{Train} \cap \text{Test} = \emptyset$.

In [ ]:
base_data_dir = Path("./colab_data")
splits = ["train", "val", "test", "unseen_test"]

# Recreate clean directory structure
if base_data_dir.exists():
    shutil.rmtree(base_data_dir)

for s in splits:
    (base_data_dir / s / "real").mkdir(parents=True, exist_ok=True)
    (base_data_dir / s / "fake").mkdir(parents=True, exist_ok=True)

rng = random.Random(SEED)
rng.shuffle(all_real_files)
rng.shuffle(all_fake_files)

# Allocation sizes per class
counts = {"train": 400, "val": 100, "test": 100, "unseen_test": 100}

cur_idx = 0
for split_name, n_per_class in counts.items():
    split_real = all_real_files[cur_idx : cur_idx + n_per_class]
    split_fake = all_fake_files[cur_idx : cur_idx + n_per_class]
    cur_idx += n_per_class
    
    for p in split_real:
        shutil.copy2(p, base_data_dir / split_name / "real" / p.name)
    for p in split_fake:
        shutil.copy2(p, base_data_dir / split_name / "fake" / p.name)
    
    print(f"[{split_name.upper():<12}] REAL: {len(split_real)} | FAKE: {len(split_fake)} -> {base_data_dir / split_name}")

print("\nZero-Leakage Audit: All partitions are 100% disjoint!")

## Step 4: TensorFlow `tf.data` Preprocessing & Augmentation Pipeline
MobileNetV2 expects input images scaled to $[-1.0, 1.0]$ via the formula:
$$x_{\text{norm}} = \frac{x}{127.5} - 1.0$$

Data augmentation (Random Horizontal Flip, Rotation, Zoom, Contrast) is applied **strictly to the training set** to prevent bias during evaluation.

In [ ]:
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 32

# Data Augmentation Layer (Training ONLY)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")

def parse_image(file_path, label):
    img_bytes = tf.io.read_file(file_path)
    img = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_HEIGHT, IMG_WIDTH), method="bicubic")
    img = (tf.cast(img, tf.float32) / 127.5) - 1.0  # MobileNetV2 normalization
    img.set_shape((IMG_HEIGHT, IMG_WIDTH, 3))
    return img, label

def create_pipeline(split_dir, is_training=False):
    real_files = [str(p) for p in (split_dir / "real").glob("*")]
    fake_files = [str(p) for p in (split_dir / "fake").glob("*")]
    paths = real_files + fake_files
    labels = [0.0] * len(real_files) + [1.0] * len(fake_files)
    
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if is_training:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
    ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    if is_training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds, len(paths)

train_ds, n_train = create_pipeline(base_data_dir / "train", is_training=True)
val_ds, n_val = create_pipeline(base_data_dir / "val", is_training=False)
test_ds, n_test = create_pipeline(base_data_dir / "test", is_training=False)
unseen_ds, n_unseen = create_pipeline(base_data_dir / "unseen_test", is_training=False)

print(f"Data Pipelines Ready: Train={n_train}, Val={n_val}, Test={n_test}, Unseen={n_unseen}")

### Visualizing Sample Augmented Training Images

In [ ]:
# Display 6 sample training images
sample_images, sample_labels = next(iter(train_ds))

plt.figure(figsize=(12, 6))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    # Un-normalize from [-1, 1] back to [0, 1] for plotting
    display_img = (sample_images[i].numpy() + 1.0) / 2.0
    plt.imshow(np.clip(display_img, 0.0, 1.0))
    lbl = "FAKE (1)" if sample_labels[i].numpy() == 1.0 else "REAL (0)"
    plt.title(f"Label: {lbl}", fontsize=11, fontweight='bold', color='red' if 'FAKE' in lbl else 'green')
    plt.axis("off")
plt.tight_layout()
plt.show()

## Step 5: MobileNetV2 Transfer Learning Architecture
1. **Backbone:** MobileNetV2 pretrained on ImageNet (154 layers), frozen initially.
2. **Feature Aggregation:** `GlobalAveragePooling2D` (reduces $7 \times 7 \times 1280 \to 1280$).
3. **Regularization:** `Dense(128, relu)` with $L_2$ regularization + `BatchNormalization` + `Dropout(0.35)`.
4. **Output Head:** Single unit with `sigmoid` activation: $\sigma(z) = \frac{1}{1 + e^{-z}}$ representing $P(y = \text{FAKE} \mid x)$.

In [ ]:
def build_mobilenetv2_detector(input_shape=(224, 224, 3), lr=1e-4):
    inputs = layers.Input(shape=input_shape, name="input_facial_crop")
    
    base_model = MobileNetV2(input_shape=input_shape, include_top=False, weights="imagenet")
    base_model.trainable = False  # Freeze pretrained weights
    
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="projection")(x)
    x = layers.BatchNormalization(name="batch_norm")(x)
    x = layers.Dropout(0.35, name="head_dropout")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="sigmoid_output")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="MobileNetV2_Deepfake_Detector")
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.5),
            tf.keras.metrics.Precision(name="precision", thresholds=0.5),
            tf.keras.metrics.Recall(name="recall", thresholds=0.5),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )
    return model, base_model

model, base_backbone = build_mobilenetv2_detector(lr=1e-4)
model.summary()

## Step 6: Phase 1 Training (Feature Extraction with Frozen Backbone)
We train the custom classification head for 5 epochs with the Adam optimizer (learning rate $10^{-4}$).
We employ `ModelCheckpoint` to save the best model weights and `ReduceLROnPlateau` for learning-rate decay.

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_mobilenetv2_deepfake.keras",
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        verbose=1,
    ),
]

print("===============================================================")
print("PHASE 1: TRAINING CLASSIFICATION HEAD (BACKBONE FROZEN)")
print("===============================================================")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks,
)

## Step 7: Phase 2 Training (Fine-Tuning Upper Convolutional Layers)
To specialize the model on subtle blending artifacts, we unfreeze the top 30 convolutional layers of MobileNetV2 and train with a smaller learning rate ($10^{-5}$) to prevent catastrophic forgetting.

In [ ]:
print("===============================================================")
print("PHASE 2: FINE-TUNING TOP 30 CONVOLUTIONAL LAYERS")
print("===============================================================")

base_backbone.trainable = True
# Freeze all layers except the last 30
for layer in base_backbone.layers[:-30]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.5),
        tf.keras.metrics.Precision(name="precision", thresholds=0.5),
        tf.keras.metrics.Recall(name="recall", thresholds=0.5),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=5,
    epochs=9,
    callbacks=callbacks,
)

## Step 8: Training & Validation Curves
Let's visualize the training progression across both Phase 1 and Phase 2.

In [ ]:
# Merge histories
acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
loss = history_phase1.history['loss'] + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy', color='#1f77b4', lw=2)
plt.plot(epochs_range, val_acc, label='Validation Accuracy', color='#ff7f0e', lw=2)
plt.axvline(x=5, color='red', linestyle='--', label='Fine-Tuning Start')
plt.title('Model Accuracy across Epochs', fontsize=13)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True, linestyle=':', alpha=0.6)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss', color='#2ca02c', lw=2)
plt.plot(epochs_range, val_loss, label='Validation Loss', color='#d62728', lw=2)
plt.axvline(x=5, color='red', linestyle='--', label='Fine-Tuning Start')
plt.title('Binary Cross-Entropy Loss across Epochs', fontsize=13)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.savefig("colab_training_curves.png", dpi=300)
plt.show()

## Step 9: Standard Test vs. Unseen Manipulation Evaluation
Now we evaluate the saved model on:
1. **Standard Test Set:** Disjoint real & fake images from the in-distribution dataset.
2. **Unseen Manipulation Set:** Out-of-distribution evaluation set.

In [ ]:
# Load best checkpoint model
best_model = tf.keras.models.load_model("best_mobilenetv2_deepfake.keras")

def evaluate_dataset(ds, split_name="Test"):
    y_true = []
    y_scores = []
    
    for imgs, lbls in ds:
        preds = best_model.predict(imgs, verbose=0)
        y_scores.extend(preds.ravel())
        y_true.extend(lbls.numpy().ravel())
        
    y_true = np.array(y_true, dtype=int)
    y_scores = np.array(y_scores, dtype=float)
    y_pred = (y_scores >= 0.5).astype(int)
    
    cm = metrics.confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    acc = (tp + tn) / len(y_true)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    auc = metrics.roc_auc_score(y_true, y_scores)
    fpr, tpr, _ = metrics.roc_curve(y_true, y_scores)
    
    print(f"\n=========================================================")
    print(f"EVALUATION RESULTS: {split_name.upper()}")
    print(f"=========================================================")
    print(f"Total Evaluated : {len(y_true)} samples (REAL={np.sum(y_true==0)}, FAKE={np.sum(y_true==1)})")
    print(f"Accuracy        : {acc * 100:.2f}%")
    print(f"Precision       : {prec * 100:.2f}%")
    print(f"Recall (Sens.)  : {rec * 100:.2f}%")
    print(f"Specificity     : {spec * 100:.2f}%")
    print(f"F1-Score        : {f1:.4f}")
    print(f"ROC-AUC         : {auc:.4f}")
    print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print(f"=========================================================\n")
    
    return {"cm": cm, "fpr": fpr, "tpr": tpr, "auc": auc, "acc": acc, "f1": f1}

std_results = evaluate_dataset(test_ds, "Standard Test Set")
unseen_results = evaluate_dataset(unseen_ds, "Unseen Manipulation Set")

### Confusion Matrix and ROC Curve Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix Heatmap
cm = std_results['cm']
im = axes[0].imshow(cm, cmap=plt.cm.Blues)
axes[0].set_title("Standard Test Set Confusion Matrix", fontsize=13, fontweight='bold')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["REAL", "FAKE"])
axes[0].set_yticklabels(["REAL", "FAKE"])
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

for i in range(2):
    for j in range(2):
        count = cm[i, j]
        pct = (count / np.sum(cm)) * 100
        axes[0].text(j, i, f"{count}\n({pct:.1f}%)", ha="center", va="center", 
                    color="white" if count > np.max(cm)/2 else "black", fontweight="bold")

# 2. ROC Curve
axes[1].plot(std_results['fpr'], std_results['tpr'], color='#1f77b4', lw=2, 
             label=f"Standard Test (AUC = {std_results['auc']:.4f})")
axes[1].plot(unseen_results['fpr'], unseen_results['tpr'], color='#ff7f0e', lw=2, linestyle='-.', 
             label=f"Unseen Test (AUC = {unseen_results['auc']:.4f})")
axes[1].plot([0, 1], [0, 1], color="navy", linestyle="--", label="Random Chance (0.50)")
axes[1].set_xlabel("False Positive Rate (1 - Specificity)")
axes[1].set_ylabel("True Positive Rate (Recall)")
axes[1].set_title("ROC Curves: Standard vs. Unseen", fontsize=13, fontweight='bold')
axes[1].legend(loc="lower right")
axes[1].grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.savefig("colab_evaluation_plots.png", dpi=300)
plt.show()

## Step 10: Interactive Image Prediction Widget
Upload any photograph from your computer or test samples to observe real-time classification!

In [ ]:
from google.colab import files
import io

print("Upload an image file (JPG or PNG) to predict:")
uploaded = files.upload()

for fn in uploaded.keys():
    pil_img = Image.open(io.BytesIO(uploaded[fn])).convert("RGB")
    resized_pil = pil_img.resize((224, 224), Image.Resampling.BICUBIC)
    
    # Preprocess
    arr = np.array(resized_pil, dtype=np.float32)
    norm_arr = (arr / 127.5) - 1.0
    batch = np.expand_dims(norm_arr, axis=0)
    
    raw_score = float(best_model.predict(batch, verbose=0)[0][0])
    is_fake = raw_score >= 0.50
    predicted_label = "FAKE" if is_fake else "REAL"
    confidence = raw_score if is_fake else (1.0 - raw_score)
    
    # Display Result
    plt.figure(figsize=(4, 4))
    plt.imshow(resized_pil)
    plt.axis("off")
    title_color = "red" if is_fake else "green"
    plt.title(f"PREDICTION: {predicted_label}\nConfidence: {confidence*100:.2f}% | Score: {raw_score:.4f}", 
              fontsize=12, fontweight="bold", color=title_color)
    plt.show()
    
    print(f"File: {fn}")
    print(f"Result: {predicted_label} (Raw Sigmoid Score: {raw_score:.4f}, Threshold: 0.50)")

## Step 11: Export & Download Trained Model
Download the saved `.keras` model artifact to your local machine.

In [ ]:
from google.colab import files

# Download the trained model
files.download("best_mobilenetv2_deepfake.keras")
print("Trained model exported successfully!")